# **Phase 3 - Vehicle Classification**

This is an open ended phase. You must build a classifier for the Stanford Cars Dataset. You can use any techniques and knowledge from Phase 1 & 2 to aid you. The ultimate goal is to get as good performance as you can on the test set.

Please follow TA instructions in lab to learn how to access the data.

# **Imports**

Import all the libraries you need for your project. Google Drive mounting code is provided if you plan to use Colab.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
import os, time, copy, math, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as transforms
from torchvision import models
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('Using device:', device, '| torch', torch.__version__, '| torchvision', torchvision.__version__)

# ------------------------------------------------------------------
# CONFIG  -- edit the paths to wherever the TA-provided data lives
# ------------------------------------------------------------------
DATA_ROOT = '/content/drive/MyDrive/stanford_cars'           # <- change me
train_images_path      = os.path.join(DATA_ROOT, 'images/train/')
train_annotations_path = os.path.join(DATA_ROOT, 'train_make.csv')
val_images_path        = os.path.join(DATA_ROOT, 'images/test/')   # labelled split we validate on
val_annotations_path   = os.path.join(DATA_ROOT, 'test_make.csv')
WEIGHTS_PATH           = os.path.join(DATA_ROOT, 'vehicle_classifier.pth')

ARCH       = 'convnext_tiny'   # 'convnext_tiny' (default) | 'resnet50' | 'efficientnet_v2_s'
IMG_SIZE   = 288               # cars are fine-grained -> a bit more resolution than 224 helps
BATCH_SIZE = 32
EPOCHS     = 25
BASE_LR    = 1e-4              # backbone lr (head gets 10x)
WEIGHT_DECAY = 0.05
LABEL_SMOOTHING = 0.1
USE_BBOX   = False             # set True only if BOTH train and test CSVs contain bounding boxes
NUM_WORKERS = 2

IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

# **1. Load Dataset**
The Stanford Dataset is provided in ```torchvision.datasets```. However, you must code your own Dataset class to load the dataset.

You will need to create your inherit from ```Dataset``` class to load the dataset and overload the ```___init__()```, ```__len__()```, and ```__getitem__()``` functions.

You will find the [following link](https://pytorch.org/tutorials/beginner/data_loading_tutorial.html) useful to help create the dataset class.

Please also create another function or method in the class: ```show_imgs()``` to help visualize the data. Similar to Phase 1 and 2, you need to display random images and label. Please display 3-5 unique classes.

In [ ]:
# Custom rotation augmentation from Phase 2 (PIL only), reused here with a small range
class RandomRotate:
    def __init__(self, min_angle=-10, max_angle=10):
        self.min_angle, self.max_angle = min_angle, max_angle
    def __call__(self, img):
        return img.rotate(random.uniform(self.min_angle, self.max_angle), resample=Image.BILINEAR)

class CarDataSet(Dataset):
    """
    Stanford Cars dataset read from an image folder + CSV annotation file.

    The CSV is auto-inspected so the same class works with the TA files (e.g. train_make.csv /
    test_make.csv): the image-filename column and label column are detected by name
    (override with file_col= / label_col= if needed). Labels may be strings (e.g. 'Audi')
    or integers (1- or 0-based). `class_names` fixes the label -> index mapping; pass the
    training set's `class_names` to the test set so both use identical indices.
    """
    FILE_KEYS  = ['filename', 'fname', 'file', 'image', 'img', 'image_name', 'path', 'relative_im_path']
    LABEL_KEYS = ['make', 'label', 'class', 'class_id', 'target', 'category', 'class_name']
    BBOX_KEYS  = [('bbox_x1', 'bbox_y1', 'bbox_x2', 'bbox_y2'), ('x1', 'y1', 'x2', 'y2'), ('xmin', 'ymin', 'xmax', 'ymax')]

    def __init__(self, images_dir, annotations_file, transform=None, class_names=None,
                 file_col=None, label_col=None, use_bbox=False, bbox_margin=0.08):
        self.images_dir = images_dir
        self.transform = transform
        self.df = pd.read_csv(annotations_file)
        cols = {c.lower().strip(): c for c in self.df.columns}

        self.file_col = file_col or next((cols[k] for k in self.FILE_KEYS if k in cols), self.df.columns[0])
        self.label_col = label_col or next((cols[k] for k in self.LABEL_KEYS if k in cols), None)

        self.bbox_cols = None
        if use_bbox:
            for keys in self.BBOX_KEYS:
                if all(k in cols for k in keys):
                    self.bbox_cols = [cols[k] for k in keys]
                    break
            if self.bbox_cols is None:
                print('WARNING: use_bbox=True but no bbox columns found -> using full images')
        self.bbox_margin = bbox_margin

        # ---- labels -> contiguous indices 0..C-1 ----
        if self.label_col is not None:
            raw = self.df[self.label_col]
            if class_names is None:
                class_names = sorted(raw.dropna().unique().tolist(), key=lambda v: (str(type(v)), v))
            self.class_names = list(class_names)
            lookup = {v: i for i, v in enumerate(self.class_names)}
            lookup.update({str(v): i for i, v in enumerate(self.class_names)})
            self.labels = [lookup.get(v, lookup.get(str(v), -1)) for v in raw]
            n_unknown = sum(l == -1 for l in self.labels)
            if n_unknown:
                print(f'WARNING: {n_unknown} labels not in class_names (marked -1)')
        else:                                        # unlabelled test set
            self.class_names = list(class_names) if class_names is not None else []
            self.labels = [-1] * len(self.df)

        self.paths = [self._resolve(f) for f in self.df[self.file_col]]

    def _resolve(self, fname):
        fname = str(fname)
        for p in (os.path.join(self.images_dir, fname), os.path.join(self.images_dir, os.path.basename(fname)), fname):
            if os.path.exists(p):
                return p
        return os.path.join(self.images_dir, os.path.basename(fname))

    def load_image(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')        # a few Stanford Cars images are greyscale
        if self.bbox_cols is not None:
            x1, y1, x2, y2 = [float(self.df.iloc[idx][c]) for c in self.bbox_cols]
            mw, mh = (x2 - x1) * self.bbox_margin, (y2 - y1) * self.bbox_margin
            img = img.crop((max(0, x1 - mw), max(0, y1 - mh), min(img.width, x2 + mw), min(img.height, y2 + mh)))
        return img

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img = self.load_image(idx)
        if self.transform is not None:
            img = self.transform(img)
        return img, self.labels[idx]

    @property
    def num_classes(self):
        return len(self.class_names)

    def label_name(self, idx):
        return str(self.class_names[idx]) if 0 <= idx < len(self.class_names) else '?'

    def show_imgs(self, n_classes=5, per_class=3):
        """Random images from `n_classes` random, distinct classes, titled with their label."""
        labels = np.array(self.labels)
        chosen = np.random.choice(np.unique(labels[labels >= 0]), size=n_classes, replace=False)
        fig, axes = plt.subplots(n_classes, per_class, figsize=(4 * per_class, 3 * n_classes))
        for r, c in enumerate(chosen):
            idxs = np.random.choice(np.where(labels == c)[0], size=per_class, replace=False)
            for k, idx in enumerate(idxs):
                ax = axes[r, k]
                ax.imshow(self.load_image(idx))
                ax.set_title(self.label_name(c), fontsize=11)
                ax.axis('off')
        plt.tight_layout()
        plt.show()

CarsDataset = CarDataSet   # alias used in Section 7 template

In [ ]:
# Transforms: strong-but-realistic augmentation for training, deterministic resize for evaluation
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.5, 1.0), ratio=(0.75, 1.33)),
    transforms.RandomHorizontalFlip(p=0.5),          # cars are left/right symmetric
    RandomRotate(-10, 10),                           # custom augmentation from Phase 2
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.15)),
])
test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Load the Data
train_data = CarDataSet(train_images_path, train_annotations_path, transform=train_transform, use_bbox=USE_BBOX)
test_data = CarDataSet(val_images_path, val_annotations_path, transform=test_transform,
                       class_names=train_data.class_names, use_bbox=USE_BBOX)
class_names = [str(c) for c in train_data.class_names]
NUM_CLASSES = train_data.num_classes
print(f'Detected columns -> file: {train_data.file_col!r}, label: {train_data.label_col!r}, bbox: {train_data.bbox_cols}')
print(f'Train: {len(train_data)} images | Val/Test: {len(test_data)} images | Classes: {NUM_CLASSES}')

# Batch Size
batch_size = BATCH_SIZE

# Import to Dataloaders
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=NUM_WORKERS,
                          pin_memory=True, drop_last=True, persistent_workers=NUM_WORKERS > 0)

test_loader = DataLoader(test_data, batch_size=batch_size * 2, shuffle=False, num_workers=NUM_WORKERS,
                         pin_memory=True, persistent_workers=NUM_WORKERS > 0)

# Class balance check
counts = pd.Series(train_data.labels).value_counts()
print(f'Images per class: min {counts.min()}, median {int(counts.median())}, max {counts.max()}')

In [ ]:
def show_imgs(dataset, n_classes=5, per_class=3):
    """Displays random images (and labels) from n_classes (3-5) unique classes."""
    dataset.show_imgs(n_classes=n_classes, per_class=per_class)

show_imgs(train_data, n_classes=5, per_class=3)

# What the network actually sees after augmentation (one image, 6 random augmentations)
idx = random.randrange(len(train_data))
fig, axes = plt.subplots(1, 6, figsize=(18, 3.2))
mean, std = torch.tensor(IMAGENET_MEAN).view(3, 1, 1), torch.tensor(IMAGENET_STD).view(3, 1, 1)
for ax in axes:
    img, lbl = train_data[idx]
    ax.imshow((img * std + mean).clamp(0, 1).permute(1, 2, 0)); ax.axis('off')
fig.suptitle(f'Training augmentations of one image ({train_data.label_name(lbl)})')
plt.tight_layout(); plt.show()

# **2. Define your Model Architecture**

It is up to you to decide which model you use. You can create your own CNN or use transfer learning.

In [ ]:
# Model definition: transfer learning from ImageNet (full fine-tuning, see report for reasoning)

def build_model(arch, num_classes, pretrained=True):
    """ImageNet-pretrained backbone with a new classification head for `num_classes`."""
    if arch == 'convnext_tiny':
        m = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1 if pretrained else None)
        m.classifier[2] = nn.Linear(m.classifier[2].in_features, num_classes)
        head = m.classifier
    elif arch == 'resnet50':
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
        m.fc = nn.Sequential(nn.Dropout(0.2), nn.Linear(m.fc.in_features, num_classes))
        head = m.fc
    elif arch == 'efficientnet_v2_s':
        m = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1 if pretrained else None)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
        head = m.classifier
    else:
        raise ValueError(arch)
    m.head_params = list(head.parameters())   # used to give the new head a larger learning rate
    return m

model = build_model(ARCH, NUM_CLASSES, pretrained=True)

# Load Model onto GPU
model = model.to(device)
n_total = sum(p.numel() for p in model.parameters())
print(f'{ARCH}: {n_total / 1e6:.1f}M parameters, {NUM_CLASSES} output classes')

# **3. Define Optimizer, Scheduler, Loss, etc.**

You chose the loss function and optimizer.

In [ ]:
# Loss Function: cross-entropy with label smoothing (fine-grained classes -> discourage over-confidence)
loss_fn = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

# Optimizer: AdamW with discriminative learning rates (pretrained backbone 1x, new head 10x)
head_ids = {id(p) for p in model.head_params}
backbone_params = [p for p in model.parameters() if id(p) not in head_ids]
optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': BASE_LR},
    {'params': model.head_params, 'lr': BASE_LR * 10},
], weight_decay=WEIGHT_DECAY)

# Scheduler: 1 epoch linear warm-up, then cosine decay to ~1% (stepped every iteration)
steps_per_epoch = len(train_loader)
warmup_steps, total_steps = steps_per_epoch, steps_per_epoch * EPOCHS
def lr_lambda(step):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * progress))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# Mixed precision (faster + less memory on Colab GPUs)
use_amp = device.type == 'cuda'
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

# **4. Train your Network**

Feel free to use your `train()` from either Phase 1 or Phase 2, whichever you feel is a better choice to understand training to tune it better. Please make sure to plot your learning curves, for both loss and accuracy! Again, this can be per iterations or epoch depending on what you see fit. You will need to save your model as `.pth` file! [This link](https://pytorch.org/tutorials/beginner/saving_loading_models.html) will be useful.

In [ ]:
# Train Network
def evaluate(model, loader, loss_fn, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad(), torch.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
        for images, labels in loader:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            outputs = model(images)
            total_loss += loss_fn(outputs.float(), labels).item() * labels.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, 100 * correct / total


def train(model, loss_fn, optimizer, scheduler, scaler, train_loader, val_loader, num_epochs, device, save_path):
    """Per-epoch train/val loss & accuracy; saves the best-val-accuracy checkpoint to save_path (.pth)."""
    history = {k: [] for k in ['train_loss', 'train_acc', 'val_loss', 'val_acc', 'lr', 'time']}
    best_acc = 0.0
    for epoch in range(num_epochs):
        start = time.time()
        model.train()
        run_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                outputs = model(images)
                loss = loss_fn(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            run_loss += loss.item() * labels.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)

        tr_loss, tr_acc = run_loss / total, 100 * correct / total
        va_loss, va_acc = evaluate(model, val_loader, loss_fn, device)
        elapsed = time.time() - start
        for k, v in zip(history, [tr_loss, tr_acc, va_loss, va_acc, optimizer.param_groups[0]['lr'], elapsed]):
            history[k].append(v)

        flag = ''
        if va_acc > best_acc:
            best_acc = va_acc
            torch.save({'state_dict': model.state_dict(), 'arch': ARCH, 'img_size': IMG_SIZE,
                        'class_names': train_data.class_names, 'use_bbox': USE_BBOX,
                        'epoch': epoch + 1, 'val_acc': va_acc}, save_path)
            flag = '  <- best, saved'
        print(f'Epoch [{epoch + 1:>2}/{num_epochs}] Train Loss {tr_loss:.4f} | Train Acc {tr_acc:6.2f}% | '
              f'Val Loss {va_loss:.4f} | Val Acc {va_acc:6.2f}% | {elapsed:5.1f}s{flag}')
    print(f'Best validation accuracy: {best_acc:.2f}%  (weights saved to {save_path})')
    return history


history = train(model, loss_fn, optimizer, scheduler, scaler, train_loader, test_loader, EPOCHS, device, WEIGHTS_PATH)

# reload the best checkpoint for evaluation
ckpt = torch.load(WEIGHTS_PATH, map_location=device)
model.load_state_dict(ckpt['state_dict'])
print(f"Loaded best checkpoint from epoch {ckpt['epoch']} (val acc {ckpt['val_acc']:.2f}%)")

In [ ]:
# Plot learning curves
def plot_learning_curves(history):
    ep = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
    axes[0].plot(ep, history['train_loss'], 'o-', label='Train'); axes[0].plot(ep, history['val_loss'], 'o-', label='Validation')
    axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch')
    axes[1].plot(ep, history['train_acc'], 'o-', label='Train'); axes[1].plot(ep, history['val_acc'], 'o-', label='Validation')
    axes[1].set_title('Accuracy (%)'); axes[1].set_xlabel('Epoch')
    axes[2].plot(ep, history['lr'], 'o-', color='gray'); axes[2].set_title('Backbone learning rate (end of epoch)')
    axes[2].set_xlabel('Epoch')
    for ax in axes[:2]:
        ax.legend()
    for ax in axes:
        ax.grid(alpha=.3)
    plt.tight_layout(); plt.show()

plot_learning_curves(history)

# **5. Evaluate**

Now we will evaluate on validation set. The TA's will run your model on the hold-out test set for your final grade.

## **5.1 Average and Per-class Accuracy**

Feel free to use `test_accuracy()` here to get accuracy. You also must show per-class accuracy. You can write your own accuracy function to get per class accuracy or use `classification_report` from `sklearn`.

In [ ]:
def get_predictions(model, loader, device):
    """Returns softmax probabilities and true labels for every sample in loader order."""
    model.eval()
    probs, labels = [], []
    with torch.no_grad(), torch.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
        for images, lbls in loader:
            probs.append(F.softmax(model(images.to(device)).float(), dim=1).cpu())
            labels.append(lbls)
    return torch.cat(probs).numpy(), torch.cat(labels).numpy()


def test_accuracy(model, loader, device):
    probs, labels = get_predictions(model, loader, device)
    preds = probs.argmax(1)
    top1 = 100 * (preds == labels).mean()
    top5 = 100 * np.mean([l in p for l, p in zip(labels, np.argsort(-probs, 1)[:, :5])])
    print(f'Average (top-1) accuracy: {top1:.2f}%   |   top-5 accuracy: {top5:.2f}%')
    return probs, labels, preds

val_probs, val_labels, val_preds = test_accuracy(model, test_loader, device)

# Per-class accuracy
print(classification_report(val_labels, val_preds, labels=list(range(NUM_CLASSES)),
                            target_names=class_names, digits=3, zero_division=0))

per_class_acc = np.array([100 * (val_preds[val_labels == c] == c).mean() if (val_labels == c).any() else np.nan
                          for c in range(NUM_CLASSES)])
order = np.argsort(per_class_acc)
plt.figure(figsize=(max(10, NUM_CLASSES * 0.12), 4.5))
plt.bar(range(NUM_CLASSES), per_class_acc[order], color='steelblue')
plt.axhline(np.nanmean(per_class_acc), color='crimson', ls='--', label=f'mean per-class acc = {np.nanmean(per_class_acc):.1f}%')
if NUM_CLASSES <= 60:
    plt.xticks(range(NUM_CLASSES), [class_names[i] for i in order], rotation=90, fontsize=8)
else:
    plt.xticks([])
plt.ylabel('Accuracy (%)'); plt.title('Per-class accuracy (sorted)'); plt.legend(); plt.tight_layout(); plt.show()
print('Hardest 10 classes:', [(class_names[i], round(float(per_class_acc[i]), 1)) for i in order[:10]])

## **5.2 Confusion Matrix**

Please plot the confusion matrix with all classess. Please make this visually readable.

In [ ]:
def plot_confusion_matrix(labels, preds, class_names):
    n = len(class_names)
    cm = confusion_matrix(labels, preds, labels=list(range(n)))
    cm_norm = cm.astype(float) / np.maximum(cm.sum(axis=1, keepdims=True), 1)   # row-normalised = recall
    size = max(10, n * 0.28)
    fig, ax = plt.subplots(figsize=(size, size * 0.9))
    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    plt.colorbar(im, ax=ax, fraction=0.035, pad=0.02, label='Fraction of true class')
    fs = max(4, min(10, 400 / n))
    ax.set_xticks(range(n)); ax.set_xticklabels(class_names, rotation=90, fontsize=fs)
    ax.set_yticks(range(n)); ax.set_yticklabels(class_names, fontsize=fs)
    ax.set_xlabel('Predicted label'); ax.set_ylabel('True label')
    ax.set_title('Normalised confusion matrix (all classes)')
    if n <= 50:                                   # annotate cells when readable
        for i in range(n):
            for j in range(n):
                if cm[i, j]:
                    ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=fs * 0.8,
                            color='white' if cm_norm[i, j] > 0.5 else 'black')
    plt.tight_layout(); plt.show()

    # most frequent confusions (off-diagonal)
    off = cm.copy(); np.fill_diagonal(off, 0)
    top = np.dstack(np.unravel_index(np.argsort(-off.ravel())[:10], off.shape))[0]
    print('Most common confusions (true -> predicted : count):')
    for i, j in top:
        if off[i, j]:
            print(f'  {class_names[i]} -> {class_names[j]} : {off[i, j]}')

plot_confusion_matrix(val_labels, val_preds, class_names)

## **5.3 Check Model Inference**

Please write a function, `show_predictions()` which will show:


1.   The top 5 confidenct predicted images with an example each.
2.   The bottom 5 confident predicted images, with an example each.

For each image, show the true label, what your model predicted and with what confidence.



In [ ]:
def show_predictions(model, dataset, loader, device, k=5, probs=None, labels=None):
    """
    Ranks classes by the model's mean confidence on images it predicts as that class, then shows
      1. the top-k most confidently predicted classes (their most confident example each)
      2. the bottom-k least confidently predicted classes (their least confident example each)
    Each image shows true label, predicted label and confidence (softmax probability).
    """
    if probs is None:
        probs, labels = get_predictions(model, loader, device)
    preds, conf = probs.argmax(1), probs.max(1)

    present = [c for c in range(probs.shape[1]) if (preds == c).any()]
    mean_conf = {c: conf[preds == c].mean() for c in present}
    ranked = sorted(present, key=lambda c: -mean_conf[c])

    def pick(classes, most_confident):
        out = []
        for c in classes:
            idxs = np.where(preds == c)[0]
            out.append(idxs[np.argmax(conf[idxs])] if most_confident else idxs[np.argmin(conf[idxs])])
        return out

    rows = [('Top-%d most confident predicted classes' % k, pick(ranked[:k], True)),
            ('Bottom-%d least confident predicted classes' % k, pick(ranked[::-1][:k], False))]
    fig, axes = plt.subplots(2, k, figsize=(4.2 * k, 9))
    for r, (title, idxs) in enumerate(rows):
        for c, idx in enumerate(idxs):
            ax = axes[r, c]
            ax.imshow(dataset.load_image(idx))
            ok = preds[idx] == labels[idx]
            ax.set_title(f'True: {dataset.label_name(labels[idx])}\nPred: {dataset.label_name(preds[idx])}\n'
                         f'Conf: {100 * conf[idx]:.1f}% (class mean {100 * mean_conf[preds[idx]]:.1f}%)',
                         fontsize=9, color='green' if ok else 'red')
            ax.axis('off')
        axes[r, 0].text(-0.08, 0.5, title, transform=axes[r, 0].transAxes, rotation=90,
                        va='center', ha='right', fontsize=11, fontweight='bold')
    plt.tight_layout(); plt.show()

show_predictions(model, test_data, test_loader, device, k=5, probs=val_probs, labels=val_labels)

## **6. Report**

Please write a report discussing all your choices and procedure to implement your vehicle classifier. In your report, include all your choices (i.e. hyperparameters, lr, models, loss, optimizer) and explain why you made those choices to achieve your performance. Your report must be thorough and comprehensive, please discuss fully how you were able to obtain a high performance. Please also include an analysis of your model, whether it may be overfitting or underfitting or fitting well and what you may have done to further improve your performance.

### Summary
I fine-tuned an ImageNet-pretrained **ConvNeXt-Tiny** end-to-end at **288×288** resolution, using strong but realistic augmentation, **AdamW** with discriminative learning rates, a **warm-up + cosine** schedule, **label smoothing**, and mixed precision. The best epoch (by validation accuracy) is saved to `vehicle_classifier.pth`.

> **Results (fill in from the outputs above after running):** best validation top-1 accuracy = **__ %**, top-5 = **__ %**, mean per-class accuracy = **__ %**, at epoch __ of 25. Final train accuracy ≈ __ %.

---

### 1. Data pipeline
- **Custom `CarDataSet`** (inherits `torch.utils.data.Dataset`). It reads the CSV annotations, auto-detects the filename and label columns (so it works with `train_make.csv` / `test_make.csv`), and maps labels to contiguous indices `0..C-1`. The test set **reuses the training set's `class_names`**, so the label→index mapping is guaranteed to be identical between training and grading. Images are converted to RGB because a few Stanford Cars images are greyscale and would otherwise crash a 3-channel network. Optional bounding-box cropping is supported but **off by default**, because the hold-out test images must be processed exactly like the training images.
- **Resolution 288.** Stanford Cars is a *fine-grained* problem: classes differ in small details such as grilles, headlights, badges and body lines. A higher input resolution keeps these details. Going from 224 to 288 is a well-known cheap boost for fine-grained tasks, while 288 still fits comfortably on a Colab GPU with batch 32 and AMP.
- **Augmentation (train only)**, chosen using what I learned in Phase 2:
  - `RandomResizedCrop(scale 0.5–1.0)`: scale and position invariance, plus partial views of the car.
  - `RandomHorizontalFlip`: cars are left/right symmetric, so a flipped car is still the same make/model.
  - My **custom `RandomRotate(±10°)` from Phase 2**: photos are taken at slightly tilted angles. The range is kept small because cars are almost never strongly rotated.
  - `ColorJitter`: the same model comes in many paint colours and lighting conditions. The network should not use colour as a shortcut.
  - `RandomErasing (p=0.25)`: simulates occlusion and forces the model to use several parts of the car instead of one discriminative patch.
  - No vertical flips or large rotations: they would create unrealistic images.
- **Evaluation transform:** deterministic `Resize((288, 288))` of the full image (no centre crop, which could cut off the front or back of the car), with ImageNet mean/std normalisation to match the pretrained weights.

### 2. Model choice
- **Transfer learning instead of a CNN from scratch.** The training set is only a few thousand images spread over many classes (roughly 40 per class). A CNN trained from scratch would massively overfit. Phase 2 showed that pretrained features give far better accuracy and much faster convergence.
- **ConvNeXt-Tiny (28M parameters)** instead of ResNet18/50. It is a modern CNN (large 7×7 depthwise kernels, LayerNorm, GELU, ImageNet top-1 ≈ 82% vs 70% for ResNet18), so its features are stronger and transfer better to fine-grained recognition, at a cost that is still manageable on Colab. `build_model()` also supports `resnet50` and `efficientnet_v2_s` for comparison.
- **Full fine-tuning (nothing frozen).** In Phase 2 (Q3.2.8–3.2.11), training only the classifier underfit badly when the target domain differed from ImageNet, and accuracy improved as more of the deeper layers were unfrozen. Fine-grained car recognition needs the high-level features to be re-shaped (ImageNet has only a few generic car classes), so all layers are trained, but the backbone uses a **10× smaller learning rate than the new head** so the pretrained features are adapted gently rather than destroyed.

### 3. Loss, optimiser, schedule and hyperparameters
| Choice | Value | Why |
|---|---|---|
| Loss | Cross-entropy, **label smoothing 0.1** | Fine-grained classes look alike and some are near-duplicates (e.g. different model years). Smoothing stops the network from becoming over-confident on the training set, which improves generalisation and calibration. |
| Optimiser | **AdamW**, weight decay 0.05 | Adam converged fastest and most reliably in Phase 1. AdamW decouples weight decay from the adaptive update, which is the standard recipe for fine-tuning ConvNeXt and gives proper L2-style regularisation. |
| Learning rate | backbone **1e-4**, head **1e-3** | The Phase 1 LR study showed that too high a rate is unstable, while too low converges slowly. A small backbone rate protects the pretrained weights; the randomly initialised head needs a larger one. |
| Schedule | 1-epoch linear **warm-up**, then **cosine** decay to 1% | Warm-up avoids large, noisy updates while the new head is still random. Cosine decay makes large progress early and fine convergence at the end. |
| Batch size | 32 | Fits 288-px images in GPU memory with AMP, and gives enough updates per epoch for a small dataset. A moderate batch also adds useful gradient noise (Phase 1, Q5.6). |
| Epochs | 25 | The validation curve has plateaued by the end of the cosine schedule. The best checkpoint is kept, so extra epochs never hurt the saved model. |
| Mixed precision | on | About 2× faster and less memory, with no accuracy loss. |
| Model selection | Save best validation accuracy | Guards against late-epoch overfitting. |

### 4. Analysis of the fit
*(Use the learning curves above. Typical behaviour for this recipe is described here; update it with your own numbers.)*
- Training accuracy rises above validation accuracy over the epochs, and training loss keeps falling while validation loss flattens. Note that with label smoothing and heavy augmentation the **training loss can't go to 0 and training accuracy is measured on augmented images**, so the gap looks smaller than the true degree of memorisation.
- If the gap stays moderate (a few percentage points) and validation accuracy plateaus instead of dropping, the model is **fitting well / mildly overfitting**. This is expected with ~40 images per class and a 28M-parameter model; the regularisation (augmentation, label smoothing, weight decay, cosine decay, best-checkpoint saving) keeps it under control.
- If validation loss starts rising clearly in later epochs, the model is overfitting. If both curves are still improving at epoch 25, it is underfitting and more epochs or a higher learning rate would help.
- **Per-class accuracy and the confusion matrix** show that the remaining errors are concentrated in visually similar classes: brands with many similar sedans/SUVs, badge-engineered twins (e.g. Chevrolet vs GMC, Dodge vs Ram), and the same model across different years. These are genuinely hard even for humans, and the most confident examples tend to be distinctive cars (supercars, unusual body styles). `show_predictions()` illustrates this: low-confidence images are usually unusual viewpoints, partial views, heavy background clutter or rare classes.

### 5. What could further improve performance
1. **Bounding-box cropping** (`USE_BBOX=True`, if bboxes are available for test images too): removes background and typically adds several points on Stanford Cars.
2. **Higher resolution** (384–448 px) or a larger backbone (ConvNeXt-Small/Base, EfficientNetV2-M, a ViT/Swin pretrained on ImageNet-21k).
3. **Stronger regularisation:** Mixup/CutMix, RandAugment/TrivialAugment, stochastic depth, EMA of the weights.
4. **Test-time augmentation** (average predictions of the image and its horizontal flip) and ensembling a few models.
5. **Class-balanced sampling or loss** if per-class accuracy shows the small makes are systematically worse.
6. A proper **hyperparameter sweep** (learning rate, weight decay, epochs) on a validation split carved out of the training data, keeping the provided test split purely for final reporting.

# **7. Loading your model code**

Please provide code to initialize your model, and code to load the model. Please call your model `vehicle_classifier`. This is so we can easily run your model.

Please also provide here your CarDataSet class and loader with option to add a path for the test data images and csv annotations.

You do not need to do anythig else, this code will be used by TAs to load and run your model to grade on test set.

[This link](https://pytorch.org/tutorials/beginner/saving_loading_models.html) will be useful.

In [ ]:
# ===================== Self-contained loading code for the TAs =====================
import os, random, math
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision import models

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

class CarDataSet(Dataset):
    """
    Stanford Cars dataset read from an image folder + CSV annotation file.

    The CSV is auto-inspected so the same class works with the TA files (e.g. train_make.csv /
    test_make.csv): the image-filename column and label column are detected by name
    (override with file_col= / label_col= if needed). Labels may be strings (e.g. 'Audi')
    or integers (1- or 0-based). `class_names` fixes the label -> index mapping; pass the
    training set's `class_names` to the test set so both use identical indices.
    """
    FILE_KEYS  = ['filename', 'fname', 'file', 'image', 'img', 'image_name', 'path', 'relative_im_path']
    LABEL_KEYS = ['make', 'label', 'class', 'class_id', 'target', 'category', 'class_name']
    BBOX_KEYS  = [('bbox_x1', 'bbox_y1', 'bbox_x2', 'bbox_y2'), ('x1', 'y1', 'x2', 'y2'), ('xmin', 'ymin', 'xmax', 'ymax')]

    def __init__(self, images_dir, annotations_file, transform=None, class_names=None,
                 file_col=None, label_col=None, use_bbox=False, bbox_margin=0.08):
        self.images_dir = images_dir
        self.transform = transform
        self.df = pd.read_csv(annotations_file)
        cols = {c.lower().strip(): c for c in self.df.columns}

        self.file_col = file_col or next((cols[k] for k in self.FILE_KEYS if k in cols), self.df.columns[0])
        self.label_col = label_col or next((cols[k] for k in self.LABEL_KEYS if k in cols), None)

        self.bbox_cols = None
        if use_bbox:
            for keys in self.BBOX_KEYS:
                if all(k in cols for k in keys):
                    self.bbox_cols = [cols[k] for k in keys]
                    break
            if self.bbox_cols is None:
                print('WARNING: use_bbox=True but no bbox columns found -> using full images')
        self.bbox_margin = bbox_margin

        # ---- labels -> contiguous indices 0..C-1 ----
        if self.label_col is not None:
            raw = self.df[self.label_col]
            if class_names is None:
                class_names = sorted(raw.dropna().unique().tolist(), key=lambda v: (str(type(v)), v))
            self.class_names = list(class_names)
            lookup = {v: i for i, v in enumerate(self.class_names)}
            lookup.update({str(v): i for i, v in enumerate(self.class_names)})
            self.labels = [lookup.get(v, lookup.get(str(v), -1)) for v in raw]
            n_unknown = sum(l == -1 for l in self.labels)
            if n_unknown:
                print(f'WARNING: {n_unknown} labels not in class_names (marked -1)')
        else:                                        # unlabelled test set
            self.class_names = list(class_names) if class_names is not None else []
            self.labels = [-1] * len(self.df)

        self.paths = [self._resolve(f) for f in self.df[self.file_col]]

    def _resolve(self, fname):
        fname = str(fname)
        for p in (os.path.join(self.images_dir, fname), os.path.join(self.images_dir, os.path.basename(fname)), fname):
            if os.path.exists(p):
                return p
        return os.path.join(self.images_dir, os.path.basename(fname))

    def load_image(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')        # a few Stanford Cars images are greyscale
        if self.bbox_cols is not None:
            x1, y1, x2, y2 = [float(self.df.iloc[idx][c]) for c in self.bbox_cols]
            mw, mh = (x2 - x1) * self.bbox_margin, (y2 - y1) * self.bbox_margin
            img = img.crop((max(0, x1 - mw), max(0, y1 - mh), min(img.width, x2 + mw), min(img.height, y2 + mh)))
        return img

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img = self.load_image(idx)
        if self.transform is not None:
            img = self.transform(img)
        return img, self.labels[idx]

    @property
    def num_classes(self):
        return len(self.class_names)

    def label_name(self, idx):
        return str(self.class_names[idx]) if 0 <= idx < len(self.class_names) else '?'

    def show_imgs(self, n_classes=5, per_class=3):
        """Random images from `n_classes` random, distinct classes, titled with their label."""
        labels = np.array(self.labels)
        chosen = np.random.choice(np.unique(labels[labels >= 0]), size=n_classes, replace=False)
        fig, axes = plt.subplots(n_classes, per_class, figsize=(4 * per_class, 3 * n_classes))
        for r, c in enumerate(chosen):
            idxs = np.random.choice(np.where(labels == c)[0], size=per_class, replace=False)
            for k, idx in enumerate(idxs):
                ax = axes[r, k]
                ax.imshow(self.load_image(idx))
                ax.set_title(self.label_name(c), fontsize=11)
                ax.axis('off')
        plt.tight_layout()
        plt.show()

CarsDataset = CarDataSet   # alias used in Section 7 template

def build_model(arch, num_classes, pretrained=True):
    """ImageNet-pretrained backbone with a new classification head for `num_classes`."""
    if arch == 'convnext_tiny':
        m = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1 if pretrained else None)
        m.classifier[2] = nn.Linear(m.classifier[2].in_features, num_classes)
        head = m.classifier
    elif arch == 'resnet50':
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
        m.fc = nn.Sequential(nn.Dropout(0.2), nn.Linear(m.fc.in_features, num_classes))
        head = m.fc
    elif arch == 'efficientnet_v2_s':
        m = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1 if pretrained else None)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
        head = m.classifier
    else:
        raise ValueError(arch)
    m.head_params = list(head.parameters())   # used to give the new head a larger learning rate
    return m


# paths
test_images_path = '/content/drive/MyDrive/stanford_cars/images/test/'         # path to images/test/
test_annotations_path = '/content/drive/MyDrive/stanford_cars/test_make.csv'   # path to test_make.csv
weights_path = '/content/drive/MyDrive/stanford_cars/vehicle_classifier.pth'   # path to your trained weights

# The checkpoint stores the architecture, input size and the class -> index mapping used in training
checkpoint = torch.load(weights_path, map_location=device)
IMG_SIZE = checkpoint['img_size']
test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

test_dataset = CarsDataset(test_images_path, test_annotations_path, transform=test_transform,
                           class_names=checkpoint['class_names'], use_bbox=checkpoint.get('use_bbox', False))
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

# Define model
vehicle_classifier = build_model(checkpoint['arch'], len(checkpoint['class_names']), pretrained=False)

# load your weights (from weights_path)
vehicle_classifier.load_state_dict(checkpoint['state_dict'])
vehicle_classifier = vehicle_classifier.to(device).eval()
print(f"Loaded {checkpoint['arch']} ({len(checkpoint['class_names'])} classes, {IMG_SIZE}px) | "
      f"test images: {len(test_dataset)}")

# Quick accuracy check on the test set
correct = total = 0
with torch.no_grad():
    for images, labels in test_loader:
        preds = vehicle_classifier(images.to(device)).argmax(1).cpu()
        mask = labels >= 0
        correct += (preds[mask] == labels[mask]).sum().item()
        total += mask.sum().item()
if total:
    print(f'Test accuracy: {100 * correct / total:.2f}%')

___
___
# End of Phase 3 😊🥳